# IW39 — Ordens de Manutenção

**Tabela:** `dev_procurement.corp_curated.tbl_ds_ind_iw39`
**Transação SAP:** IW39 · **Colunas:** 61
**Clustering declarado:** `cod_ordem`

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros criados na barra de widgets do notebook.
SELECT
  :f_cod_centro_planejamento_manutencao AS f_cod_centro_planejamento_manutencao,
  :f_cod_empresa AS f_cod_empresa,
  :f_tp_ordem AS f_tp_ordem;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_ind_iw39
WHERE (:f_cod_centro_planejamento_manutencao = '' OR `cod_centro_planejamento_manutencao` = :f_cod_centro_planejamento_manutencao)
  AND (:f_cod_empresa = '' OR `cod_empresa` = :f_cod_empresa)
  AND (:f_tp_ordem = '' OR `tp_ordem` = :f_tp_ordem);

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_ind_iw39;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_ind_iw39;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_ind_iw39 LIMIT 20;

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'cod_ordem' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_ordem` FROM base)
UNION ALL
SELECT 'cod_ordem + num_nota' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_ordem`, `num_nota` FROM base)
UNION ALL
SELECT 'cod_ordem + cod_equipamento' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_ordem`, `cod_equipamento` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'cod_ordem' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_ordem`, COUNT(*) AS qtd FROM base GROUP BY `cod_ordem` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_ordem + num_nota' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_ordem`, `num_nota`, COUNT(*) AS qtd FROM base GROUP BY `cod_ordem`, `num_nota` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_ordem + cod_equipamento' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_ordem`, `cod_equipamento`, COUNT(*) AS qtd FROM base GROUP BY `cod_ordem`, `cod_equipamento` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(61,
    'cod_ordem', 'string', COUNT_IF(`cod_ordem` IS NULL), COUNT_IF(`cod_ordem` IS NOT NULL AND lower(trim(`cod_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_planejamento_mnt', 'string', COUNT_IF(`tp_grupo_planejamento_mnt` IS NULL), COUNT_IF(`tp_grupo_planejamento_mnt` IS NOT NULL AND lower(trim(`tp_grupo_planejamento_mnt`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_planejamento_mnt`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_trabalho_principal', 'string', COUNT_IF(`cod_centro_trabalho_principal` IS NULL), COUNT_IF(`cod_centro_trabalho_principal` IS NOT NULL AND lower(trim(`cod_centro_trabalho_principal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_trabalho_principal`) RLIKE '^0+([.,]0+)?$'),
    'tp_ordem', 'string', COUNT_IF(`tp_ordem` IS NULL), COUNT_IF(`tp_ordem` IS NOT NULL AND lower(trim(`tp_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_prioridade', 'string', COUNT_IF(`cod_prioridade` IS NULL), COUNT_IF(`cod_prioridade` IS NOT NULL AND lower(trim(`cod_prioridade`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_prioridade`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'string', COUNT_IF(`dt_criacao` IS NULL), COUNT_IF(`dt_criacao` IS NOT NULL AND lower(trim(`dt_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_base_inicio', 'string', COUNT_IF(`dt_base_inicio` IS NULL), COUNT_IF(`dt_base_inicio` IS NOT NULL AND lower(trim(`dt_base_inicio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^0+([.,]0+)?$'),
    'dt_base_fim', 'string', COUNT_IF(`dt_base_fim` IS NULL), COUNT_IF(`dt_base_fim` IS NOT NULL AND lower(trim(`dt_base_fim`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_base_fim`) RLIKE '^0+([.,]0+)?$'),
    'num_nota', 'string', COUNT_IF(`num_nota` IS NULL), COUNT_IF(`num_nota` IS NOT NULL AND lower(trim(`num_nota`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_nota`) RLIKE '^0+([.,]0+)?$'),
    'desc_breve', 'string', COUNT_IF(`desc_breve` IS NULL), COUNT_IF(`desc_breve` IS NOT NULL AND lower(trim(`desc_breve`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_breve`) RLIKE '^0+([.,]0+)?$'),
    'cod_revisao', 'string', COUNT_IF(`cod_revisao` IS NULL), COUNT_IF(`cod_revisao` IS NOT NULL AND lower(trim(`cod_revisao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_revisao`) RLIKE '^0+([.,]0+)?$'),
    'cod_localizacao', 'string', COUNT_IF(`cod_localizacao` IS NULL), COUNT_IF(`cod_localizacao` IS NOT NULL AND lower(trim(`cod_localizacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_localizacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_processamento', 'string', COUNT_IF(`tp_grupo_processamento` IS NULL), COUNT_IF(`tp_grupo_processamento` IS NOT NULL AND lower(trim(`tp_grupo_processamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_processamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_usuario_criacao', 'string', COUNT_IF(`cod_usuario_criacao` IS NULL), COUNT_IF(`cod_usuario_criacao` IS NOT NULL AND lower(trim(`cod_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'vl_custo_total_planejado', 'double', COUNT_IF(`vl_custo_total_planejado` IS NULL), 0L, COUNT_IF(`vl_custo_total_planejado` = 0),
    'cod_usuario_modificacao', 'string', COUNT_IF(`cod_usuario_modificacao` IS NULL), COUNT_IF(`cod_usuario_modificacao` IS NOT NULL AND lower(trim(`cod_usuario_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_plano_manutencao', 'string', COUNT_IF(`cod_plano_manutencao` IS NULL), COUNT_IF(`cod_plano_manutencao` IS NOT NULL AND lower(trim(`cod_plano_manutencao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_plano_manutencao`) RLIKE '^0+([.,]0+)?$'),
    'cod_equipamento', 'string', COUNT_IF(`cod_equipamento` IS NULL), COUNT_IF(`cod_equipamento` IS NOT NULL AND lower(trim(`cod_equipamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_equipamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo_responsavel', 'string', COUNT_IF(`cod_centro_custo_responsavel` IS NULL), COUNT_IF(`cod_centro_custo_responsavel` IS NOT NULL AND lower(trim(`cod_centro_custo_responsavel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo_responsavel`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'dt_encerramento_tecnico', 'string', COUNT_IF(`dt_encerramento_tecnico` IS NULL), COUNT_IF(`dt_encerramento_tecnico` IS NOT NULL AND lower(trim(`dt_encerramento_tecnico`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^0+([.,]0+)?$'),
    'dt_inicio_programado', 'string', COUNT_IF(`dt_inicio_programado` IS NULL), COUNT_IF(`dt_inicio_programado` IS NOT NULL AND lower(trim(`dt_inicio_programado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^0+([.,]0+)?$'),
    'dt_fim_programado', 'string', COUNT_IF(`dt_fim_programado` IS NULL), COUNT_IF(`dt_fim_programado` IS NOT NULL AND lower(trim(`dt_fim_programado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^0+([.,]0+)?$'),
    'dt_referencia', 'string', COUNT_IF(`dt_referencia` IS NULL), COUNT_IF(`dt_referencia` IS NOT NULL AND lower(trim(`dt_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_referencia`) RLIKE '^0+([.,]0+)?$'),
    'dh_referencia', 'string', COUNT_IF(`dh_referencia` IS NULL), COUNT_IF(`dh_referencia` IS NOT NULL AND lower(trim(`dh_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_referencia`) RLIKE '^0+([.,]0+)?$'),
    'dt_ultima_modificacao', 'string', COUNT_IF(`dt_ultima_modificacao` IS NULL), COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_liberacao_real', 'string', COUNT_IF(`dt_liberacao_real` IS NULL), COUNT_IF(`dt_liberacao_real` IS NOT NULL AND lower(trim(`dt_liberacao_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^0+([.,]0+)?$'),
    'dt_fim_confirmado_ordem', 'string', COUNT_IF(`dt_fim_confirmado_ordem` IS NULL), COUNT_IF(`dt_fim_confirmado_ordem` IS NOT NULL AND lower(trim(`dt_fim_confirmado_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^0+([.,]0+)?$'),
    'dh_base_fim', 'string', COUNT_IF(`dh_base_fim` IS NULL), COUNT_IF(`dh_base_fim` IS NOT NULL AND lower(trim(`dh_base_fim`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_base_fim`) RLIKE '^0+([.,]0+)?$'),
    'dh_fim_confirmado_ordem', 'string', COUNT_IF(`dh_fim_confirmado_ordem` IS NULL), COUNT_IF(`dh_fim_confirmado_ordem` IS NOT NULL AND lower(trim(`dh_fim_confirmado_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_fim_confirmado_ordem`) RLIKE '^0+([.,]0+)?$'),
    'dt_inicio_real', 'string', COUNT_IF(`dt_inicio_real` IS NULL), COUNT_IF(`dt_inicio_real` IS NOT NULL AND lower(trim(`dt_inicio_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^0+([.,]0+)?$'),
    'dh_inicio_real', 'string', COUNT_IF(`dh_inicio_real` IS NULL), COUNT_IF(`dh_inicio_real` IS NOT NULL AND lower(trim(`dh_inicio_real`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_inicio_real`) RLIKE '^0+([.,]0+)?$'),
    'dh_base_inicio', 'string', COUNT_IF(`dh_base_inicio` IS NULL), COUNT_IF(`dh_base_inicio` IS NOT NULL AND lower(trim(`dh_base_inicio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dh_base_inicio`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'cod_pep_ordem', 'string', COUNT_IF(`cod_pep_ordem` IS NULL), COUNT_IF(`cod_pep_ordem` IS NOT NULL AND lower(trim(`cod_pep_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_pep_ordem`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_ordem', 'string', COUNT_IF(`tp_categoria_ordem` IS NULL), COUNT_IF(`tp_categoria_ordem` IS NOT NULL AND lower(trim(`tp_categoria_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_planejamento_manutencao', 'string', COUNT_IF(`cod_centro_planejamento_manutencao` IS NULL), COUNT_IF(`cod_centro_planejamento_manutencao` IS NOT NULL AND lower(trim(`cod_centro_planejamento_manutencao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_planejamento_manutencao`) RLIKE '^0+([.,]0+)?$'),
    'cod_id_objeto_centro_trabalho', 'string', COUNT_IF(`cod_id_objeto_centro_trabalho` IS NULL), COUNT_IF(`cod_id_objeto_centro_trabalho` IS NOT NULL AND lower(trim(`cod_id_objeto_centro_trabalho`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_id_objeto_centro_trabalho`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_lista_operacoes', 'string', COUNT_IF(`tp_grupo_lista_operacoes` IS NULL), COUNT_IF(`tp_grupo_lista_operacoes` IS NOT NULL AND lower(trim(`tp_grupo_lista_operacoes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_lista_operacoes`) RLIKE '^0+([.,]0+)?$'),
    'cod_variante_lista_operacoes', 'string', COUNT_IF(`cod_variante_lista_operacoes` IS NULL), COUNT_IF(`cod_variante_lista_operacoes` IS NOT NULL AND lower(trim(`cod_variante_lista_operacoes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_variante_lista_operacoes`) RLIKE '^0+([.,]0+)?$'),
    'num_serie', 'string', COUNT_IF(`num_serie` IS NULL), COUNT_IF(`num_serie` IS NOT NULL AND lower(trim(`num_serie`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_serie`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_produto_material', 'string', COUNT_IF(`desc_produto_material` IS NULL), COUNT_IF(`desc_produto_material` IS NOT NULL AND lower(trim(`desc_produto_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_produto_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_conjunto', 'string', COUNT_IF(`cod_conjunto` IS NULL), COUNT_IF(`cod_conjunto` IS NOT NULL AND lower(trim(`cod_conjunto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conjunto`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'cod_esquema_calculo_custos', 'string', COUNT_IF(`cod_esquema_calculo_custos` IS NULL), COUNT_IF(`cod_esquema_calculo_custos` IS NOT NULL AND lower(trim(`cod_esquema_calculo_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_esquema_calculo_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_divisao', 'string', COUNT_IF(`cod_divisao` IS NULL), COUNT_IF(`cod_divisao` IS NOT NULL AND lower(trim(`cod_divisao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_divisao`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_lucro', 'string', COUNT_IF(`cod_centro_lucro` IS NULL), COUNT_IF(`cod_centro_lucro` IS NOT NULL AND lower(trim(`cod_centro_lucro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_lucro`) RLIKE '^0+([.,]0+)?$'),
    'cod_area_contabilidade_custos', 'string', COUNT_IF(`cod_area_contabilidade_custos` IS NULL), COUNT_IF(`cod_area_contabilidade_custos` IS NOT NULL AND lower(trim(`cod_area_contabilidade_custos`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_area_contabilidade_custos`) RLIKE '^0+([.,]0+)?$'),
    'cod_ordem_cliente', 'string', COUNT_IF(`cod_ordem_cliente` IS NULL), COUNT_IF(`cod_ordem_cliente` IS NOT NULL AND lower(trim(`cod_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem_cliente`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_venda', 'string', COUNT_IF(`num_item_pedido_venda` IS NULL), COUNT_IF(`num_item_pedido_venda` IS NOT NULL AND lower(trim(`num_item_pedido_venda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_venda`) RLIKE '^0+([.,]0+)?$'),
    'cod_diagrama_rede_rede_superior', 'string', COUNT_IF(`cod_diagrama_rede_rede_superior` IS NULL), COUNT_IF(`cod_diagrama_rede_rede_superior` IS NOT NULL AND lower(trim(`cod_diagrama_rede_rede_superior`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_diagrama_rede_rede_superior`) RLIKE '^0+([.,]0+)?$'),
    'cod_ordem_tem_texto_descritivo', 'string', COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NULL), COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NOT NULL AND lower(trim(`cod_ordem_tem_texto_descritivo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ordem_tem_texto_descritivo`) RLIKE '^0+([.,]0+)?$'),
    'ind_marcacao_eliminacao', 'string', COUNT_IF(`ind_marcacao_eliminacao` IS NULL), COUNT_IF(`ind_marcacao_eliminacao` IS NOT NULL AND lower(trim(`ind_marcacao_eliminacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_marcacao_eliminacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_rua_endereco', 'string', COUNT_IF(`cod_rua_endereco` IS NULL), COUNT_IF(`cod_rua_endereco` IS NOT NULL AND lower(trim(`cod_rua_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_rua_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_regiao_endereco', 'string', COUNT_IF(`cod_regiao_endereco` IS NULL), COUNT_IF(`cod_regiao_endereco` IS NOT NULL AND lower(trim(`cod_regiao_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_regiao_endereco`) RLIKE '^0+([.,]0+)?$'),
    'nm_cidade_endereco', 'string', COUNT_IF(`nm_cidade_endereco` IS NULL), COUNT_IF(`nm_cidade_endereco` IS NOT NULL AND lower(trim(`nm_cidade_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_cidade_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_pais_endereco', 'string', COUNT_IF(`cod_pais_endereco` IS NULL), COUNT_IF(`cod_pais_endereco` IS NOT NULL AND lower(trim(`cod_pais_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_pais_endereco`) RLIKE '^0+([.,]0+)?$'),
    'cod_postal_endereco', 'string', COUNT_IF(`cod_postal_endereco` IS NULL), COUNT_IF(`cod_postal_endereco` IS NOT NULL AND lower(trim(`cod_postal_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_postal_endereco`) RLIKE '^0+([.,]0+)?$'),
    'num_telefone_endereco', 'string', COUNT_IF(`num_telefone_endereco` IS NULL), COUNT_IF(`num_telefone_endereco` IS NOT NULL AND lower(trim(`num_telefone_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_telefone_endereco`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(61,
    'cod_ordem', 'string', approx_count_distinct(`cod_ordem`),
    'tp_grupo_planejamento_mnt', 'string', approx_count_distinct(`tp_grupo_planejamento_mnt`),
    'cod_centro_trabalho_principal', 'string', approx_count_distinct(`cod_centro_trabalho_principal`),
    'tp_ordem', 'string', approx_count_distinct(`tp_ordem`),
    'cod_prioridade', 'string', approx_count_distinct(`cod_prioridade`),
    'dt_criacao', 'string', approx_count_distinct(`dt_criacao`),
    'dt_base_inicio', 'string', approx_count_distinct(`dt_base_inicio`),
    'dt_base_fim', 'string', approx_count_distinct(`dt_base_fim`),
    'num_nota', 'string', approx_count_distinct(`num_nota`),
    'desc_breve', 'string', approx_count_distinct(`desc_breve`),
    'cod_revisao', 'string', approx_count_distinct(`cod_revisao`),
    'cod_localizacao', 'string', approx_count_distinct(`cod_localizacao`),
    'tp_grupo_processamento', 'string', approx_count_distinct(`tp_grupo_processamento`),
    'cod_usuario_criacao', 'string', approx_count_distinct(`cod_usuario_criacao`),
    'vl_custo_total_planejado', 'double', approx_count_distinct(`vl_custo_total_planejado`),
    'cod_usuario_modificacao', 'string', approx_count_distinct(`cod_usuario_modificacao`),
    'cod_plano_manutencao', 'string', approx_count_distinct(`cod_plano_manutencao`),
    'cod_equipamento', 'string', approx_count_distinct(`cod_equipamento`),
    'cod_centro_custo_responsavel', 'string', approx_count_distinct(`cod_centro_custo_responsavel`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'dt_encerramento_tecnico', 'string', approx_count_distinct(`dt_encerramento_tecnico`),
    'dt_inicio_programado', 'string', approx_count_distinct(`dt_inicio_programado`),
    'dt_fim_programado', 'string', approx_count_distinct(`dt_fim_programado`),
    'dt_referencia', 'string', approx_count_distinct(`dt_referencia`),
    'dh_referencia', 'string', approx_count_distinct(`dh_referencia`),
    'dt_ultima_modificacao', 'string', approx_count_distinct(`dt_ultima_modificacao`),
    'dt_liberacao_real', 'string', approx_count_distinct(`dt_liberacao_real`),
    'dt_fim_confirmado_ordem', 'string', approx_count_distinct(`dt_fim_confirmado_ordem`),
    'dh_base_fim', 'string', approx_count_distinct(`dh_base_fim`),
    'dh_fim_confirmado_ordem', 'string', approx_count_distinct(`dh_fim_confirmado_ordem`),
    'dt_inicio_real', 'string', approx_count_distinct(`dt_inicio_real`),
    'dh_inicio_real', 'string', approx_count_distinct(`dh_inicio_real`),
    'dh_base_inicio', 'string', approx_count_distinct(`dh_base_inicio`),
    'cod_elemento_pep', 'string', approx_count_distinct(`cod_elemento_pep`),
    'cod_pep_ordem', 'string', approx_count_distinct(`cod_pep_ordem`),
    'tp_categoria_ordem', 'string', approx_count_distinct(`tp_categoria_ordem`),
    'cod_centro_planejamento_manutencao', 'string', approx_count_distinct(`cod_centro_planejamento_manutencao`),
    'cod_id_objeto_centro_trabalho', 'string', approx_count_distinct(`cod_id_objeto_centro_trabalho`),
    'tp_grupo_lista_operacoes', 'string', approx_count_distinct(`tp_grupo_lista_operacoes`),
    'cod_variante_lista_operacoes', 'string', approx_count_distinct(`cod_variante_lista_operacoes`),
    'num_serie', 'string', approx_count_distinct(`num_serie`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_produto_material', 'string', approx_count_distinct(`desc_produto_material`),
    'cod_conjunto', 'string', approx_count_distinct(`cod_conjunto`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'cod_esquema_calculo_custos', 'string', approx_count_distinct(`cod_esquema_calculo_custos`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_divisao', 'string', approx_count_distinct(`cod_divisao`),
    'cod_centro_lucro', 'string', approx_count_distinct(`cod_centro_lucro`),
    'cod_area_contabilidade_custos', 'string', approx_count_distinct(`cod_area_contabilidade_custos`),
    'cod_ordem_cliente', 'string', approx_count_distinct(`cod_ordem_cliente`),
    'num_item_pedido_venda', 'string', approx_count_distinct(`num_item_pedido_venda`),
    'cod_diagrama_rede_rede_superior', 'string', approx_count_distinct(`cod_diagrama_rede_rede_superior`),
    'cod_ordem_tem_texto_descritivo', 'string', approx_count_distinct(`cod_ordem_tem_texto_descritivo`),
    'ind_marcacao_eliminacao', 'string', approx_count_distinct(`ind_marcacao_eliminacao`),
    'cod_rua_endereco', 'string', approx_count_distinct(`cod_rua_endereco`),
    'cod_regiao_endereco', 'string', approx_count_distinct(`cod_regiao_endereco`),
    'nm_cidade_endereco', 'string', approx_count_distinct(`nm_cidade_endereco`),
    'cod_pais_endereco', 'string', approx_count_distinct(`cod_pais_endereco`),
    'cod_postal_endereco', 'string', approx_count_distinct(`cod_postal_endereco`),
    'num_telefone_endereco', 'string', approx_count_distinct(`num_telefone_endereco`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'tp_grupo_planejamento_mnt' AS coluna, CAST(`tp_grupo_planejamento_mnt` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_grupo_planejamento_mnt` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro_trabalho_principal' AS coluna, CAST(`cod_centro_trabalho_principal` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro_trabalho_principal` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_ordem' AS coluna, CAST(`tp_ordem` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_ordem` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_prioridade' AS coluna, CAST(`cod_prioridade` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_prioridade` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_revisao' AS coluna, CAST(`cod_revisao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_revisao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_processamento' AS coluna, CAST(`tp_grupo_processamento` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_grupo_processamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_categoria_ordem' AS coluna, CAST(`tp_categoria_ordem` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_categoria_ordem` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_divisao' AS coluna, CAST(`cod_divisao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_divisao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro_planejamento_manutencao' AS coluna, CAST(`cod_centro_planejamento_manutencao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro_planejamento_manutencao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_marcacao_eliminacao' AS coluna, CAST(`ind_marcacao_eliminacao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_marcacao_eliminacao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(1,
    'vl_custo_total_planejado', 'double', COUNT(`vl_custo_total_planejado`), CAST(MIN(`vl_custo_total_planejado`) AS DOUBLE), CAST(MAX(`vl_custo_total_planejado`) AS DOUBLE), CAST(AVG(`vl_custo_total_planejado`) AS DOUBLE), CAST(percentile_approx(`vl_custo_total_planejado`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_custo_total_planejado`, 0.95) AS DOUBLE), COUNT_IF(`vl_custo_total_planejado` < 0), COUNT_IF(`vl_custo_total_planejado` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10. Datas armazenadas como STRING

**Armadilha conhecida:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
É a mesma data em formato diferente — já gerou **16.773 falsos positivos em outra transação**.

Se a coluna `veredito` acusar mais de um formato, a normalização é obrigatória.

In [0]:
-- 10. DATAS ARMAZENADAS COMO STRING
-- ARMADILHA: SAP exporta '2024-02-23 00:00:00', Datalake grava '20240223'.
-- Mesma data, formato diferente. Ja gerou 16.773 falsos positivos.
WITH t AS (SELECT COUNT(*) AS total FROM base),
d AS (
  SELECT stack(11,
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL OR trim(`dt_criacao`) = ''), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_criacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_criacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END), MAX(CASE WHEN trim(`dt_criacao`) NOT IN ('', '00000000') THEN `dt_criacao` END),
    'dt_base_inicio', COUNT_IF(`dt_base_inicio` IS NULL OR trim(`dt_base_inicio`) = ''), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_base_inicio`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_base_inicio`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_base_inicio`) NOT IN ('', '00000000') THEN `dt_base_inicio` END), MAX(CASE WHEN trim(`dt_base_inicio`) NOT IN ('', '00000000') THEN `dt_base_inicio` END),
    'dt_base_fim', COUNT_IF(`dt_base_fim` IS NULL OR trim(`dt_base_fim`) = ''), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_base_fim`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_base_fim`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_base_fim`) NOT IN ('', '00000000') THEN `dt_base_fim` END), MAX(CASE WHEN trim(`dt_base_fim`) NOT IN ('', '00000000') THEN `dt_base_fim` END),
    'dt_encerramento_tecnico', COUNT_IF(`dt_encerramento_tecnico` IS NULL OR trim(`dt_encerramento_tecnico`) = ''), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_encerramento_tecnico`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_encerramento_tecnico`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_encerramento_tecnico`) NOT IN ('', '00000000') THEN `dt_encerramento_tecnico` END), MAX(CASE WHEN trim(`dt_encerramento_tecnico`) NOT IN ('', '00000000') THEN `dt_encerramento_tecnico` END),
    'dt_inicio_programado', COUNT_IF(`dt_inicio_programado` IS NULL OR trim(`dt_inicio_programado`) = ''), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_inicio_programado`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_inicio_programado`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_inicio_programado`) NOT IN ('', '00000000') THEN `dt_inicio_programado` END), MAX(CASE WHEN trim(`dt_inicio_programado`) NOT IN ('', '00000000') THEN `dt_inicio_programado` END),
    'dt_fim_programado', COUNT_IF(`dt_fim_programado` IS NULL OR trim(`dt_fim_programado`) = ''), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_fim_programado`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_fim_programado`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_fim_programado`) NOT IN ('', '00000000') THEN `dt_fim_programado` END), MAX(CASE WHEN trim(`dt_fim_programado`) NOT IN ('', '00000000') THEN `dt_fim_programado` END),
    'dt_referencia', COUNT_IF(`dt_referencia` IS NULL OR trim(`dt_referencia`) = ''), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_referencia`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_referencia`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_referencia`) NOT IN ('', '00000000') THEN `dt_referencia` END), MAX(CASE WHEN trim(`dt_referencia`) NOT IN ('', '00000000') THEN `dt_referencia` END),
    'dt_ultima_modificacao', COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = ''), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_ultima_modificacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END), MAX(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END),
    'dt_liberacao_real', COUNT_IF(`dt_liberacao_real` IS NULL OR trim(`dt_liberacao_real`) = ''), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_liberacao_real`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_liberacao_real`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_liberacao_real`) NOT IN ('', '00000000') THEN `dt_liberacao_real` END), MAX(CASE WHEN trim(`dt_liberacao_real`) NOT IN ('', '00000000') THEN `dt_liberacao_real` END),
    'dt_fim_confirmado_ordem', COUNT_IF(`dt_fim_confirmado_ordem` IS NULL OR trim(`dt_fim_confirmado_ordem`) = ''), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_fim_confirmado_ordem`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_fim_confirmado_ordem`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_fim_confirmado_ordem`) NOT IN ('', '00000000') THEN `dt_fim_confirmado_ordem` END), MAX(CASE WHEN trim(`dt_fim_confirmado_ordem`) NOT IN ('', '00000000') THEN `dt_fim_confirmado_ordem` END),
    'dt_inicio_real', COUNT_IF(`dt_inicio_real` IS NULL OR trim(`dt_inicio_real`) = ''), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_inicio_real`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_inicio_real`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_inicio_real`) NOT IN ('', '00000000') THEN `dt_inicio_real` END), MAX(CASE WHEN trim(`dt_inicio_real`) NOT IN ('', '00000000') THEN `dt_inicio_real` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM base
)
SELECT d.coluna, d.vazios, d.fmt_AAAAMMDD, d.fmt_ISO, d.fmt_BR, d.data_zero,
       t.total - d.vazios - d.fmt_AAAAMMDD - d.fmt_ISO - d.fmt_BR AS nao_reconhecido,
       d.minimo, d.maximo,
       CASE WHEN (CASE WHEN d.fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato na mesma coluna'
            ELSE 'formato unico' END AS veredito
FROM d CROSS JOIN t
ORDER BY d.coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(8,
    'cod_ordem', 'string', COUNT_IF(CAST(`cod_ordem` AS STRING) IS NULL OR trim(CAST(`cod_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`cod_ordem` AS STRING)))), MAX(length(trim(CAST(`cod_ordem` AS STRING)))), COUNT_IF(trim(CAST(`cod_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_ordem` AS STRING) <> trim(CAST(`cod_ordem` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_ordem` AS STRING)), '^0+', '')),
    'num_nota', 'string', COUNT_IF(CAST(`num_nota` AS STRING) IS NULL OR trim(CAST(`num_nota` AS STRING)) = ''), MIN(length(trim(CAST(`num_nota` AS STRING)))), MAX(length(trim(CAST(`num_nota` AS STRING)))), COUNT_IF(trim(CAST(`num_nota` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_nota` AS STRING) <> trim(CAST(`num_nota` AS STRING))), COUNT(DISTINCT trim(CAST(`num_nota` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_nota` AS STRING)), '^0+', '')),
    'cod_equipamento', 'string', COUNT_IF(CAST(`cod_equipamento` AS STRING) IS NULL OR trim(CAST(`cod_equipamento` AS STRING)) = ''), MIN(length(trim(CAST(`cod_equipamento` AS STRING)))), MAX(length(trim(CAST(`cod_equipamento` AS STRING)))), COUNT_IF(trim(CAST(`cod_equipamento` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_equipamento` AS STRING) <> trim(CAST(`cod_equipamento` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_equipamento` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_equipamento` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro_custo` AS STRING) <> trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'cod_centro_custo_responsavel', 'string', COUNT_IF(CAST(`cod_centro_custo_responsavel` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo_responsavel` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo_responsavel` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo_responsavel` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo_responsavel` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro_custo_responsavel` AS STRING) <> trim(CAST(`cod_centro_custo_responsavel` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro_custo_responsavel` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo_responsavel` AS STRING)), '^0+', '')),
    'cod_plano_manutencao', 'string', COUNT_IF(CAST(`cod_plano_manutencao` AS STRING) IS NULL OR trim(CAST(`cod_plano_manutencao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_plano_manutencao` AS STRING)))), MAX(length(trim(CAST(`cod_plano_manutencao` AS STRING)))), COUNT_IF(trim(CAST(`cod_plano_manutencao` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_plano_manutencao` AS STRING) <> trim(CAST(`cod_plano_manutencao` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_plano_manutencao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_plano_manutencao` AS STRING)), '^0+', '')),
    'num_serie', 'string', COUNT_IF(CAST(`num_serie` AS STRING) IS NULL OR trim(CAST(`num_serie` AS STRING)) = ''), MIN(length(trim(CAST(`num_serie` AS STRING)))), MAX(length(trim(CAST(`num_serie` AS STRING)))), COUNT_IF(trim(CAST(`num_serie` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_serie` AS STRING) <> trim(CAST(`num_serie` AS STRING))), COUNT(DISTINCT trim(CAST(`num_serie` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_serie` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro_planejamento_manutencao
SELECT `cod_centro_planejamento_manutencao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro_planejamento_manutencao`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_empresa
SELECT `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_empresa`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR tp_ordem
SELECT `tp_ordem`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `tp_ordem`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **cod_ordem**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `cod_ordem`, COUNT(*) AS qtd
FROM base
GROUP BY `cod_ordem`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `cod_ordem` FROM base GROUP BY `cod_ordem` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`cod_ordem`)
),
agg AS (
  SELECT `cod_ordem`,
         COUNT(DISTINCT `tp_grupo_planejamento_mnt`) AS `tp_grupo_planejamento_mnt`,
         COUNT(DISTINCT `cod_centro_trabalho_principal`) AS `cod_centro_trabalho_principal`,
         COUNT(DISTINCT `tp_ordem`) AS `tp_ordem`,
         COUNT(DISTINCT `cod_prioridade`) AS `cod_prioridade`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `dt_base_inicio`) AS `dt_base_inicio`,
         COUNT(DISTINCT `dt_base_fim`) AS `dt_base_fim`,
         COUNT(DISTINCT `num_nota`) AS `num_nota`,
         COUNT(DISTINCT `desc_breve`) AS `desc_breve`,
         COUNT(DISTINCT `cod_revisao`) AS `cod_revisao`,
         COUNT(DISTINCT `cod_localizacao`) AS `cod_localizacao`,
         COUNT(DISTINCT `tp_grupo_processamento`) AS `tp_grupo_processamento`,
         COUNT(DISTINCT `cod_usuario_criacao`) AS `cod_usuario_criacao`,
         COUNT(DISTINCT `vl_custo_total_planejado`) AS `vl_custo_total_planejado`,
         COUNT(DISTINCT `cod_usuario_modificacao`) AS `cod_usuario_modificacao`,
         COUNT(DISTINCT `cod_plano_manutencao`) AS `cod_plano_manutencao`,
         COUNT(DISTINCT `cod_equipamento`) AS `cod_equipamento`,
         COUNT(DISTINCT `cod_centro_custo_responsavel`) AS `cod_centro_custo_responsavel`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `dt_encerramento_tecnico`) AS `dt_encerramento_tecnico`,
         COUNT(DISTINCT `dt_inicio_programado`) AS `dt_inicio_programado`,
         COUNT(DISTINCT `dt_fim_programado`) AS `dt_fim_programado`,
         COUNT(DISTINCT `dt_referencia`) AS `dt_referencia`,
         COUNT(DISTINCT `dh_referencia`) AS `dh_referencia`,
         COUNT(DISTINCT `dt_ultima_modificacao`) AS `dt_ultima_modificacao`,
         COUNT(DISTINCT `dt_liberacao_real`) AS `dt_liberacao_real`,
         COUNT(DISTINCT `dt_fim_confirmado_ordem`) AS `dt_fim_confirmado_ordem`,
         COUNT(DISTINCT `dh_base_fim`) AS `dh_base_fim`,
         COUNT(DISTINCT `dh_fim_confirmado_ordem`) AS `dh_fim_confirmado_ordem`,
         COUNT(DISTINCT `dt_inicio_real`) AS `dt_inicio_real`,
         COUNT(DISTINCT `dh_inicio_real`) AS `dh_inicio_real`,
         COUNT(DISTINCT `dh_base_inicio`) AS `dh_base_inicio`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `cod_pep_ordem`) AS `cod_pep_ordem`,
         COUNT(DISTINCT `tp_categoria_ordem`) AS `tp_categoria_ordem`,
         COUNT(DISTINCT `cod_centro_planejamento_manutencao`) AS `cod_centro_planejamento_manutencao`,
         COUNT(DISTINCT `cod_id_objeto_centro_trabalho`) AS `cod_id_objeto_centro_trabalho`,
         COUNT(DISTINCT `tp_grupo_lista_operacoes`) AS `tp_grupo_lista_operacoes`,
         COUNT(DISTINCT `cod_variante_lista_operacoes`) AS `cod_variante_lista_operacoes`,
         COUNT(DISTINCT `num_serie`) AS `num_serie`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_produto_material`) AS `desc_produto_material`,
         COUNT(DISTINCT `cod_conjunto`) AS `cod_conjunto`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `cod_esquema_calculo_custos`) AS `cod_esquema_calculo_custos`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_divisao`) AS `cod_divisao`,
         COUNT(DISTINCT `cod_centro_lucro`) AS `cod_centro_lucro`,
         COUNT(DISTINCT `cod_area_contabilidade_custos`) AS `cod_area_contabilidade_custos`,
         COUNT(DISTINCT `cod_ordem_cliente`) AS `cod_ordem_cliente`,
         COUNT(DISTINCT `num_item_pedido_venda`) AS `num_item_pedido_venda`,
         COUNT(DISTINCT `cod_diagrama_rede_rede_superior`) AS `cod_diagrama_rede_rede_superior`,
         COUNT(DISTINCT `cod_ordem_tem_texto_descritivo`) AS `cod_ordem_tem_texto_descritivo`,
         COUNT(DISTINCT `ind_marcacao_eliminacao`) AS `ind_marcacao_eliminacao`,
         COUNT(DISTINCT `cod_rua_endereco`) AS `cod_rua_endereco`,
         COUNT(DISTINCT `cod_regiao_endereco`) AS `cod_regiao_endereco`,
         COUNT(DISTINCT `nm_cidade_endereco`) AS `nm_cidade_endereco`,
         COUNT(DISTINCT `cod_pais_endereco`) AS `cod_pais_endereco`,
         COUNT(DISTINCT `cod_postal_endereco`) AS `cod_postal_endereco`,
         COUNT(DISTINCT `num_telefone_endereco`) AS `num_telefone_endereco`
  FROM d GROUP BY `cod_ordem`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(60,
    'tp_grupo_planejamento_mnt', MAX(`tp_grupo_planejamento_mnt`),
    'cod_centro_trabalho_principal', MAX(`cod_centro_trabalho_principal`),
    'tp_ordem', MAX(`tp_ordem`),
    'cod_prioridade', MAX(`cod_prioridade`),
    'dt_criacao', MAX(`dt_criacao`),
    'dt_base_inicio', MAX(`dt_base_inicio`),
    'dt_base_fim', MAX(`dt_base_fim`),
    'num_nota', MAX(`num_nota`),
    'desc_breve', MAX(`desc_breve`),
    'cod_revisao', MAX(`cod_revisao`),
    'cod_localizacao', MAX(`cod_localizacao`),
    'tp_grupo_processamento', MAX(`tp_grupo_processamento`),
    'cod_usuario_criacao', MAX(`cod_usuario_criacao`),
    'vl_custo_total_planejado', MAX(`vl_custo_total_planejado`),
    'cod_usuario_modificacao', MAX(`cod_usuario_modificacao`),
    'cod_plano_manutencao', MAX(`cod_plano_manutencao`),
    'cod_equipamento', MAX(`cod_equipamento`),
    'cod_centro_custo_responsavel', MAX(`cod_centro_custo_responsavel`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'dt_encerramento_tecnico', MAX(`dt_encerramento_tecnico`),
    'dt_inicio_programado', MAX(`dt_inicio_programado`),
    'dt_fim_programado', MAX(`dt_fim_programado`),
    'dt_referencia', MAX(`dt_referencia`),
    'dh_referencia', MAX(`dh_referencia`),
    'dt_ultima_modificacao', MAX(`dt_ultima_modificacao`),
    'dt_liberacao_real', MAX(`dt_liberacao_real`),
    'dt_fim_confirmado_ordem', MAX(`dt_fim_confirmado_ordem`),
    'dh_base_fim', MAX(`dh_base_fim`),
    'dh_fim_confirmado_ordem', MAX(`dh_fim_confirmado_ordem`),
    'dt_inicio_real', MAX(`dt_inicio_real`),
    'dh_inicio_real', MAX(`dh_inicio_real`),
    'dh_base_inicio', MAX(`dh_base_inicio`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'cod_pep_ordem', MAX(`cod_pep_ordem`),
    'tp_categoria_ordem', MAX(`tp_categoria_ordem`),
    'cod_centro_planejamento_manutencao', MAX(`cod_centro_planejamento_manutencao`),
    'cod_id_objeto_centro_trabalho', MAX(`cod_id_objeto_centro_trabalho`),
    'tp_grupo_lista_operacoes', MAX(`tp_grupo_lista_operacoes`),
    'cod_variante_lista_operacoes', MAX(`cod_variante_lista_operacoes`),
    'num_serie', MAX(`num_serie`),
    'cod_material', MAX(`cod_material`),
    'desc_produto_material', MAX(`desc_produto_material`),
    'cod_conjunto', MAX(`cod_conjunto`),
    'cod_moeda', MAX(`cod_moeda`),
    'cod_esquema_calculo_custos', MAX(`cod_esquema_calculo_custos`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_divisao', MAX(`cod_divisao`),
    'cod_centro_lucro', MAX(`cod_centro_lucro`),
    'cod_area_contabilidade_custos', MAX(`cod_area_contabilidade_custos`),
    'cod_ordem_cliente', MAX(`cod_ordem_cliente`),
    'num_item_pedido_venda', MAX(`num_item_pedido_venda`),
    'cod_diagrama_rede_rede_superior', MAX(`cod_diagrama_rede_rede_superior`),
    'cod_ordem_tem_texto_descritivo', MAX(`cod_ordem_tem_texto_descritivo`),
    'ind_marcacao_eliminacao', MAX(`ind_marcacao_eliminacao`),
    'cod_rua_endereco', MAX(`cod_rua_endereco`),
    'cod_regiao_endereco', MAX(`cod_regiao_endereco`),
    'nm_cidade_endereco', MAX(`nm_cidade_endereco`),
    'cod_pais_endereco', MAX(`cod_pais_endereco`),
    'cod_postal_endereco', MAX(`cod_postal_endereco`),
    'num_telefone_endereco', MAX(`num_telefone_endereco`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
-- Esta tabela NAO possui coluna de data de ingestao.
-- ACAO: solicitar ao time de dados a inclusao de dateingest ou equivalente.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_ind_iw39 LIMIT 10;

## 16. Análises específicas — IW39

### 16.1 Coerência do fluxo de datas
A ordem tem sequência lógica: criação → início → fim → encerramento técnico.
Datas fora de sequência indicam erro de carga ou reprogramação legítima.

In [0]:
-- 16.1 COERENCIA DO FLUXO DE DATAS
SELECT 'criacao <= base_inicio' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' AND `dt_criacao` <> '00000000' THEN `dt_criacao` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') IS NOT NULL) AS ambas_preenchidas,
       COUNT_IF(to_date(CASE WHEN `dt_criacao` RLIKE '^[0-9]{8}$' AND `dt_criacao` <> '00000000' THEN `dt_criacao` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd')) AS violacoes
  FROM base
UNION ALL
SELECT 'base_inicio <= base_fim' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd') IS NOT NULL) AS ambas_preenchidas,
       COUNT_IF(to_date(CASE WHEN `dt_base_inicio` RLIKE '^[0-9]{8}$' AND `dt_base_inicio` <> '00000000' THEN `dt_base_inicio` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd')) AS violacoes
  FROM base
UNION ALL
SELECT 'inicio_prog <= fim_prog' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_programado` RLIKE '^[0-9]{8}$' AND `dt_inicio_programado` <> '00000000' THEN `dt_inicio_programado` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_fim_programado` RLIKE '^[0-9]{8}$' AND `dt_fim_programado` <> '00000000' THEN `dt_fim_programado` END, 'yyyyMMdd') IS NOT NULL) AS ambas_preenchidas,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_programado` RLIKE '^[0-9]{8}$' AND `dt_inicio_programado` <> '00000000' THEN `dt_inicio_programado` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_fim_programado` RLIKE '^[0-9]{8}$' AND `dt_fim_programado` <> '00000000' THEN `dt_fim_programado` END, 'yyyyMMdd')) AS violacoes
  FROM base
UNION ALL
SELECT 'inicio_real <= fim_confirmado' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_real` RLIKE '^[0-9]{8}$' AND `dt_inicio_real` <> '00000000' THEN `dt_inicio_real` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_fim_confirmado_ordem` RLIKE '^[0-9]{8}$' AND `dt_fim_confirmado_ordem` <> '00000000' THEN `dt_fim_confirmado_ordem` END, 'yyyyMMdd') IS NOT NULL) AS ambas_preenchidas,
       COUNT_IF(to_date(CASE WHEN `dt_inicio_real` RLIKE '^[0-9]{8}$' AND `dt_inicio_real` <> '00000000' THEN `dt_inicio_real` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_fim_confirmado_ordem` RLIKE '^[0-9]{8}$' AND `dt_fim_confirmado_ordem` <> '00000000' THEN `dt_fim_confirmado_ordem` END, 'yyyyMMdd')) AS violacoes
  FROM base
UNION ALL
SELECT 'base_fim <= encerr_tecnico' AS regra,
       COUNT_IF(to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd') IS NOT NULL AND to_date(CASE WHEN `dt_encerramento_tecnico` RLIKE '^[0-9]{8}$' AND `dt_encerramento_tecnico` <> '00000000' THEN `dt_encerramento_tecnico` END, 'yyyyMMdd') IS NOT NULL) AS ambas_preenchidas,
       COUNT_IF(to_date(CASE WHEN `dt_base_fim` RLIKE '^[0-9]{8}$' AND `dt_base_fim` <> '00000000' THEN `dt_base_fim` END, 'yyyyMMdd') > to_date(CASE WHEN `dt_encerramento_tecnico` RLIKE '^[0-9]{8}$' AND `dt_encerramento_tecnico` <> '00000000' THEN `dt_encerramento_tecnico` END, 'yyyyMMdd')) AS violacoes
  FROM base
ORDER BY violacoes DESC;

### 16.2 Campo `cod_revisao` — Grande Parada vs Recorrente
Campo usado operacionalmente para distinguir material de Grande Parada (GP)
de manutenção Recorrente (REC).

In [0]:
-- 16.2 COD_REVISAO
SELECT COALESCE(NULLIF(trim(cod_revisao), ''), '(vazio)') AS cod_revisao,
       COUNT(*) AS ordens,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       ROUND(SUM(COALESCE(vl_custo_total_planejado, 0)), 2) AS custo_total
FROM base
GROUP BY COALESCE(NULLIF(trim(cod_revisao), ''), '(vazio)')
ORDER BY ordens DESC
LIMIT 40;

### 16.3 Completude de vínculos

In [0]:
-- 16.3 COMPLETUDE DE VINCULOS
SELECT 'num_nota' AS vinculo,
       COUNT_IF(`num_nota` IS NULL OR trim(`num_nota`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`num_nota` IS NULL OR trim(`num_nota`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_equipamento' AS vinculo,
       COUNT_IF(`cod_equipamento` IS NULL OR trim(`cod_equipamento`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_equipamento` IS NULL OR trim(`cod_equipamento`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_centro_custo' AS vinculo,
       COUNT_IF(`cod_centro_custo` IS NULL OR trim(`cod_centro_custo`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_centro_custo` IS NULL OR trim(`cod_centro_custo`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_centro_custo_responsavel' AS vinculo,
       COUNT_IF(`cod_centro_custo_responsavel` IS NULL OR trim(`cod_centro_custo_responsavel`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_centro_custo_responsavel` IS NULL OR trim(`cod_centro_custo_responsavel`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_material' AS vinculo,
       COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_plano_manutencao' AS vinculo,
       COUNT_IF(`cod_plano_manutencao` IS NULL OR trim(`cod_plano_manutencao`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_plano_manutencao` IS NULL OR trim(`cod_plano_manutencao`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_elemento_pep' AS vinculo,
       COUNT_IF(`cod_elemento_pep` IS NULL OR trim(`cod_elemento_pep`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_elemento_pep` IS NULL OR trim(`cod_elemento_pep`) = '') / COUNT(*), 2) AS pct
  FROM base
UNION ALL
SELECT 'cod_localizacao' AS vinculo,
       COUNT_IF(`cod_localizacao` IS NULL OR trim(`cod_localizacao`) = '') AS sem_vinculo,
       ROUND(100.0 * COUNT_IF(`cod_localizacao` IS NULL OR trim(`cod_localizacao`) = '') / COUNT(*), 2) AS pct
  FROM base
ORDER BY pct DESC;

### 16.4 Teste dirigido — colunas de endereço
As 8 últimas colunas do schema não possuem comentário no `DESCRIBE` — assinatura
típica de coluna nunca carregada. Esta consulta confirma ou descarta a suspeita.

In [0]:
-- 16.4 COLUNAS DE ENDERECO
SELECT 'cod_ordem_tem_texto_descritivo' AS coluna,
       COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NOT NULL AND trim(`cod_ordem_tem_texto_descritivo`) <> '') AS uteis,
       COUNT(DISTINCT `cod_ordem_tem_texto_descritivo`) AS distintos,
       CASE WHEN COUNT_IF(`cod_ordem_tem_texto_descritivo` IS NOT NULL AND trim(`cod_ordem_tem_texto_descritivo`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'ind_marcacao_eliminacao' AS coluna,
       COUNT_IF(`ind_marcacao_eliminacao` IS NOT NULL AND trim(`ind_marcacao_eliminacao`) <> '') AS uteis,
       COUNT(DISTINCT `ind_marcacao_eliminacao`) AS distintos,
       CASE WHEN COUNT_IF(`ind_marcacao_eliminacao` IS NOT NULL AND trim(`ind_marcacao_eliminacao`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'cod_rua_endereco' AS coluna,
       COUNT_IF(`cod_rua_endereco` IS NOT NULL AND trim(`cod_rua_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `cod_rua_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`cod_rua_endereco` IS NOT NULL AND trim(`cod_rua_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'cod_regiao_endereco' AS coluna,
       COUNT_IF(`cod_regiao_endereco` IS NOT NULL AND trim(`cod_regiao_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `cod_regiao_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`cod_regiao_endereco` IS NOT NULL AND trim(`cod_regiao_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'nm_cidade_endereco' AS coluna,
       COUNT_IF(`nm_cidade_endereco` IS NOT NULL AND trim(`nm_cidade_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `nm_cidade_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`nm_cidade_endereco` IS NOT NULL AND trim(`nm_cidade_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'cod_pais_endereco' AS coluna,
       COUNT_IF(`cod_pais_endereco` IS NOT NULL AND trim(`cod_pais_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `cod_pais_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`cod_pais_endereco` IS NOT NULL AND trim(`cod_pais_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'cod_postal_endereco' AS coluna,
       COUNT_IF(`cod_postal_endereco` IS NOT NULL AND trim(`cod_postal_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `cod_postal_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`cod_postal_endereco` IS NOT NULL AND trim(`cod_postal_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
UNION ALL
SELECT 'num_telefone_endereco' AS coluna,
       COUNT_IF(`num_telefone_endereco` IS NOT NULL AND trim(`num_telefone_endereco`) <> '') AS uteis,
       COUNT(DISTINCT `num_telefone_endereco`) AS distintos,
       CASE WHEN COUNT_IF(`num_telefone_endereco` IS NOT NULL AND trim(`num_telefone_endereco`) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END AS veredito
  FROM base
ORDER BY uteis, coluna;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- 90. INTEGRIDADE: cod_material -> dev_procurement.corp_curated.tbl_ds_mdm_mm60.cod_material
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM base WHERE `cod_material` IS NOT NULL AND trim(CAST(`cod_material` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '61', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       'cod_ordem', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_ordem' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_ordem` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_ordem + num_nota' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_ordem`, `num_nota` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_ordem + cod_equipamento' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_ordem`, `cod_equipamento` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
